# Limpieza de datos

Partimos de `Property.csv` (191.393 anuncios de zameen.com) y dejamos un conjunto limpio de viviendas en venta.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
df = pd.read_csv("../data/bronze/Property.csv", sep = ";")

In [3]:
df.head(3)

,property_id,location_id,page_url,property_type,price,location,city,province_name,latitude,longitude,baths,area,purpose,bedrooms,date_added,agency,agent
0,347795,8,https://www.zameen.com/Property/lahore_model_t...,House,220000000,Model Town,Lahore,Punjab,31.483869,74.325686,0,6 Kanal,For Sale,0,07-17-2019,Real Biz International,Usama Khan
1,482892,48,https://www.zameen.com/Property/lahore_multan_...,House,40000000,Multan Road,Lahore,Punjab,31.431593,74.179980,5,1 Kanal,For Sale,5,10-06-2018,Khan Estate,mohsinkhan and B
2,555962,75,https://www.zameen.com/Property/eden_eden_aven...,House,9500000,Eden,Lahore,Punjab,31.499348,74.416959,0,9 Marla,For Sale,3,07-03-2019,Shahum Estate 2,"Babar Hameed, Raja Omar"


## 1. Columnas que no describen la vivienda

- `property_id` y `page_url` son identificadores del anuncio.
- `agency` y `agent` son de quién publica, no de la vivienda.
- `province_name` se deduce de la ciudad (cada ciudad está en una sola provincia), así que es redundante.

Las quitamos antes de buscar duplicados: dos anuncios iguales con distinto `property_id` son el mismo anuncio.

In [4]:
df = df.drop(columns=["property_id", "page_url", "agency", "agent", "province_name"])

In [5]:
print(f"Número de filas en el dataset original: {len(df)}")
n_antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicados exactos eliminados: {n_antes - len(df)}")

Número de filas en el dataset original: 191393
Duplicados exactos eliminados: 22758


## 2. Solo anuncios de venta

El alquiler es otro problema: precio mensual y en otra escala. Filtramos **primero** por venta para que las cifras de los pasos siguientes se refieran solo a ventas.

In [6]:
print(df["purpose"].value_counts())
n_antes = len(df)
df = df[df["purpose"] == "For Sale"].reset_index(drop=True)
print(f"Anuncios de alquiler eliminados: {n_antes - len(df)}")
df = df.drop(columns=["purpose"])

purpose
For Sale    111797
For Rent     56838
Name: count, dtype: int64
Anuncios de alquiler eliminados: 56838


Estudiamos la variable "baths":

In [7]:
df["baths"].value_counts()

baths
0      26114
3      19717
2      14872
4      14375
6      13779
5      12532
7       4670
1       3053
8       1653
10       539
9        476
12         7
11         6
13         2
403        1
14         1
Name: count, dtype: int64

Decidimos restringir nuestro problema a viviendas con 12 baños como máximo.

In [8]:
n_antes = len(df)
df = df[df["baths"] <= 12].reset_index(drop=True)
print(f"Filas eliminadas con más 12 baños: {n_antes - len(df)}")

Filas eliminadas con más 12 baños: 4


Estudiamos la variable "bedrooms": 

In [9]:
df["bedrooms"].value_counts()

bedrooms
3     27607
5     19891
4     18132
2     17274
0     11973
6      9598
7      2430
1      2357
8      1176
9       589
10      461
11      266
12       20
15        6
16        4
14        2
13        2
25        2
27        2
18        1
Name: count, dtype: int64

Decidimos restringir nuestro problema a viviendas con 12 habitaciones como máximo.

In [10]:
n_antes = len(df)
df = df[df["bedrooms"] <= 12].reset_index(drop=True)
print(f"Filas eliminadas com más de 12 habitaciones: {n_antes - len(df)}")

Filas eliminadas com más de 12 habitaciones: 19


Estudiamos las variables "bedroom" - "bathroom" conjuntas: 

In [11]:
df.groupby("bedrooms")["baths"].value_counts().unstack(fill_value=0)

baths,0,1,2,3,4,5,6,7,8,9,10,11,12
bedrooms,,,,,,,,,,,,,
0,11726,39,65,43,27,24,31,6,5,5,2,0,0
1,363,1709,272,9,2,2,0,0,0,0,0,0,0
2,2484,1097,11833,1707,111,15,15,5,4,0,3,0,0
3,4053,154,1798,15997,4943,611,38,5,2,1,5,0,0
4,2677,37,672,1089,8002,4677,881,77,9,5,6,0,0
5,2474,7,140,490,728,6512,7858,1594,73,7,8,0,0
6,1574,7,71,267,338,429,4287,2109,434,62,20,0,0
7,264,0,10,59,99,108,282,745,729,94,40,0,0
8,200,0,6,32,81,78,197,68,327,109,77,0,1


## 3. Interpretamos valores ceros en "baths" y "bedrooms":

1. Comparamos el precio de casas del mismo tamaño (10 marla y 1 kanal, los más comunes) con y sin baños.
2. Miramos si el porcentaje de ceros cambia según la ciudad.

In [12]:
casas = df[(df["property_type"] == "House") & (df["area"].isin(["10 Marla", "1 Kanal"]))]
print(casas.groupby(["area", casas["baths"] == 0])["price"].median())

print((df["baths"] == 0).groupby(df["city"]).mean().round(2))

area      baths
1 Kanal   False    46000000.0
          True     46500000.0
10 Marla  False    22000000.0
          True     20000000.0
Name: price, dtype: float64
city
Faisalabad    0.57
Islamabad     0.23
Karachi       0.23
Lahore        0.19
Rawalpindi    0.29
Name: baths, dtype: float64


## Decisión:

- Las casas con 0 baños cuestan prácticamente lo mismo que las del mismo tamaño que sí indican los baños. Una casa sin baño sería mucho más barata, así que el 0 no es un valor real sino un dato que no se rellenó.
- Eliminamos los anuncios con 0 baños o 0 habitaciones. Imputarlos supondría inventarnos el valor en casi una cuarta parte de los anuncios, y el modelo aprendería en parte de datos que no son reales. Preferimos trabajar solo con anuncios que tienen la información completa.
- El coste es perder casi una cuarta parte de las ventas, y no por igual en todas las ciudades (Faisalabad pierde más de la mitad). Por eso en la partición estratificamos por ciudad.

In [13]:
n_antes = len(df)
df = df[(df["baths"] > 0) & (df["bedrooms"] > 0)].reset_index(drop=True)
print(f"Filas eliminadas con 0 baños o 0 habitaciones: {n_antes - len(df)}")

Filas eliminadas con 0 baños o 0 habitaciones: 26344


Estudiamos la variable "city":

In [14]:
df["city"].value_counts()

city
Lahore        32795
Karachi       30620
Islamabad     11988
Rawalpindi     8230
Faisalabad     1797
Name: count, dtype: int64

Estudiamos la variable "location": 

In [15]:
df["location"].value_counts()

location
DHA Defence               12672
Bahria Town                4776
Bahria Town Karachi        4603
Bahria Town Rawalpindi     3253
Gulistan-e-Jauhar          2965
                          ...  
Nabha Road                    1
Lakshmi Chowk                 1
FDA City                      1
Defence Fort                  1
Sultan Colony                 1
Name: count, Length: 1395, dtype: int64

Hay 1.460 zonas, muy desiguales, pues unas pocas concentran miles de anuncios y muchas tienen solo uno. Un one-hot daría más de mil columnas, así que obtamos por codificarlo a partir del precio medio de cada zona, conocido como target encoding. Esto se hará después de la partición, ajustando solo con el train.

## 4. Ubicación: distancia al centro y coordenadas erróneas

Calculamos la distancia en línea recta de cada anuncio al centro de su ciudad con la fórmula de Haversine. Nos sirve como variable para el modelo y para detectar coordenadas que no son de esa ciudad.

Conservamos `latitude` y `longitude`: son de la subzona, no de cada vivienda, pero pueden servir al modelo para captar zonas cercanas entre sí.

In [16]:
CENTROS = {
    "Lahore":     (31.582045, 74.329376),
    "Karachi":    (24.860966, 66.990501),
    "Islamabad":  (33.738045, 73.084488),
    "Rawalpindi": (33.626057, 73.071442),
    "Faisalabad": (31.418715, 73.079109),
}
R_TIERRA_KM = 6371.0088

lat_c = df["city"].map(lambda c: CENTROS[c][0])
lon_c = df["city"].map(lambda c: CENTROS[c][1])

lat1, lon1 = np.radians(df["latitude"]), np.radians(df["longitude"])
lat2, lon2 = np.radians(lat_c), np.radians(lon_c)

# Fórmula de Haversine (distancia en línea recta sobre la esfera, en km)
a = np.sin((lat1 - lat2) / 2) ** 2 + np.cos(lat2) * np.cos(lat1) * np.sin((lon1 - lon2) / 2) ** 2
df["dist_centro_km"] = (2 * R_TIERRA_KM * np.arcsin(np.sqrt(a))).round(2)

df.groupby("city")["dist_centro_km"].describe()

,count,mean,std,min,25%,50%,75%,max
city,,,,,,,,
Faisalabad,1797.0,8.490128,92.701623,0.00,3.96,5.470,7.42,3932.06
Islamabad,11988.0,16.229965,56.954447,0.72,10.61,14.740,20.52,4386.21
Karachi,30620.0,16.375048,14.712882,0.30,9.29,13.270,19.72,942.89
Lahore,32795.0,16.274881,7.392096,0.32,12.25,15.400,19.39,268.65
Rawalpindi,8230.0,10.036103,44.768002,0.27,6.83,8.835,12.03,4046.89


In [17]:
n_antes = len(df)
df = df[df["dist_centro_km"] <= 100].reset_index(drop=True)
print(f"Filas eliminadas por coordenadas erróneas: {n_antes - len(df)}")

Filas eliminadas por coordenadas erróneas: 16


## 5. Precio en euros

Pasamos de rupias a euros con el cambio del 18/07/2019 (1 € = 179,61 PKR), dentro del periodo de los anuncios. Al ser una constante no cambia nada del modelo, pero hace los precios más fáciles de interpretar.

Una vivienda en venta por menos de 1.000 € no es real y es error de carga.

In [18]:
PKR_TO_EUR_2019 = 1 / 179.61   # ≈ 0.005568, valutafx, 18/07/2019

df["price"] = (df["price"] * PKR_TO_EUR_2019).round(2)

In [19]:
n_antes = len(df)
df = df[df["price"] >= 1000].reset_index(drop=True)
print(f"Filas eliminadas por precio < 1.000 €: {n_antes - len(df)}")

Filas eliminadas por precio < 1.000 €: 79


## 6. Superficie en m²

La superficie viene como texto en unidades locales (`"4.8 Marla"`, `"1 Kanal"`). La convertimos nosotros a m², respetando los decimales:

- 1 marla = 272,25 ft² ≈ 25,29 m²
- 1 kanal = 20 marla ≈ 505,86 m²

In [20]:
df[["area_num", "area_unit"]] = df["area"].str.split(" ", expand=True)
df["area_num"] = df["area_num"].str.replace(",", "", regex=False).astype(float)

df["area_unit"].value_counts()

area_unit
Marla    70976
Kanal    14359
Name: count, dtype: int64

In [21]:
FT2_TO_M2 = 0.09290304
FACTORES_M2 = {
    "Marla": 272.25 * FT2_TO_M2,        # 1 marla = 272,25 ft² ≈ 25,29 m²
    "Kanal": 20 * 272.25 * FT2_TO_M2,   # 1 kanal = 20 marlas ≈ 505,86 m²
}

df["Total_Area"] = (df["area_num"] * df["area_unit"].map(FACTORES_M2)).round(2)
df = df.drop(columns=["area", "area_num", "area_unit"])

n_antes = len(df)
df = df[df["Total_Area"] >= 10].reset_index(drop=True)
print(f"Filas eliminadas por superficie < 10 m²: {n_antes - len(df)}")
df["Total_Area"].describe().round(1)

Filas eliminadas por superficie < 10 m²: 13


count      85322.0
mean         350.8
std        12037.0
min           10.1
25%          121.4
50%          177.0
75%          270.6
max      2276356.7
Name: Total_Area, dtype: float64

## 7. Tipo de vivienda

Miramos cuántos anuncios hay de cada tipo, su superficie y su precio por m².

In [22]:
df["property_type"].value_counts()

property_type
House            63206
Flat             19404
Upper Portion     1617
Lower Portion      663
Farm House         211
Penthouse          208
Room                13
Name: count, dtype: int64

In [23]:
eur_m2 = df["price"] / df["Total_Area"]
(df.assign(eur_m2=eur_m2)
   .groupby("property_type")
   .agg(anuncios=("price", "size"), m2_mediana=("Total_Area", "median"), eur_m2_mediana=("eur_m2", "median"))
   .sort_values("anuncios", ascending=False)
   .round(0))

,anuncios,m2_mediana,eur_m2_mediana
property_type,,,
House,63206,202.0,484.0
Flat,19404,124.0,344.0
Upper Portion,1617,139.0,332.0
Lower Portion,663,142.0,336.0
Farm House,211,3541.0,94.0
Penthouse,208,228.0,316.0
Room,13,126.0,440.0


- "Upper Portion", "Lower Portion" y "Penthouse" son plantas de un edificio o de una casa. Tienen un precio por m² casi igual que "Flat" (unos 330 €/m² frente a 340) y muy distinto de "House" (unos 480). "Room" son solo 23 anuncios. Los agrupamos todos en "Flat", pues así el modelo tiene dos categorías con muchos datos en vez de cinco muy desequilibradas.
- "Farm House" son fincas de unos 3.000 m² a 70 €/m². Se trata de otro mercado distintoy apenas hay 600, así pues las eliminamos

In [24]:
n_antes = len(df)
df = df[df["property_type"] != "Farm House"].reset_index(drop=True)
print(f"Farm House eliminadas: {n_antes - len(df)}")

agrupar = {"Upper Portion": "Flat", "Lower Portion": "Flat", "Penthouse": "Flat", "Room": "Flat"}
df["property_type"] = df["property_type"].replace(agrupar)
df["property_type"].value_counts()

Farm House eliminadas: 211


property_type
House    63206
Flat     21905
Name: count, dtype: int64

## Superficies imposibles según el tipo

Una superficie errónea (por ejemplo, poner *kanal* en vez de *marla*, ×20) se delata en el precio por m²: el precio es el de una vivienda normal pero la superficie es enorme. Lo miramos por tramos.

In [25]:
df["eur_m2"] = df["price"] / df["Total_Area"]

tramos = {
    "Flat":  [0, 100, 200, 300, 400, 600, 1000, np.inf],
    "House": [0, 100, 500, 1000, 2500, 5000, np.inf],
}
for tipo, cortes in tramos.items():
    sub = df[df["property_type"] == tipo]
    tabla = sub.groupby(pd.cut(sub["Total_Area"], cortes), observed=True)["eur_m2"].agg(["size", "median"]).round(0)
    print(f"\n{tipo}\n{tabla}")


Flat
                  size  median
Total_Area                    
(0.0, 100.0]      5616   285.0
(100.0, 200.0]   10767   347.0
(200.0, 300.0]    4162   393.0
(300.0, 400.0]     948   423.0
(400.0, 600.0]     283   485.0
(600.0, 1000.0]     62    35.0
(1000.0, inf]       67    42.0

House
                   size  median
Total_Area                     
(0.0, 100.0]       6736   426.0
(100.0, 500.0]    42496   477.0
(500.0, 1000.0]   11834   523.0
(1000.0, 2500.0]   1985   710.0
(2500.0, 5000.0]     65   605.0
(5000.0, inf]        90    29.0


## Decisiones: 

- Pisos: hasta 600 m² el precio por m² es coherente. Por encima cae a unos 40 €/m²: son errores de unidad. Quitamos los pisos de más de 600 m².
- Casas: hasta 5.000 m² el precio por m² es coherente. A partir de 10 kanal (5.058 m²) cae a unos 30 €/m²: casi todas son "10 Kanal" con precio de una casa de 10 marla. Quitamos las casas de 5.000 m² o más.

In [26]:
n_antes = len(df)
df = df[~((df["property_type"] == "Flat") & (df["Total_Area"] > 600))].reset_index(drop=True)
print(f"Pisos eliminados (> 600 m²): {n_antes - len(df)}")

n_antes = len(df)
df = df[~((df["property_type"] == "House") & (df["Total_Area"] >= 5000))].reset_index(drop=True)
print(f"Casas eliminadas (>= 5.000 m²): {n_antes - len(df)}")
df = df.drop(columns=["eur_m2"])

Pisos eliminados (> 600 m²): 129
Casas eliminadas (>= 5.000 m²): 90


## 8. Anuncios repetidos

- "date_added" es la fecha de la última actualización. Los agentes vuelven a publicar o actualizan la misma vivienda, y en el archivo aparece varias veces con todo igual menos la fecha. Nos quedamos con la versión más reciente de cada uno.

- Quedan además anuncios que probablemente son la misma vivienda con algún detalle cambiado. Esos no se pueden quitar con seguridad; los tratamos en la partición, agrupándolos para que no caigan unos en train y otros en test.

In [27]:
df["date_added"] = pd.to_datetime(df["date_added"], format="%m-%d-%Y")

columnas_sin_fecha = df.columns.drop("date_added")
df = df.sort_values("date_added")
n_antes = len(df)
df = df.drop_duplicates(subset=columnas_sin_fecha, keep="last").reset_index(drop=True)
print(f"Repetidos eliminados (todo igual salvo la fecha): {n_antes - len(df)}")

Repetidos eliminados (todo igual salvo la fecha): 20170


## 9. Resumen y guardado

In [28]:
df.head(3)

,location_id,property_type,price,location,city,latitude,longitude,baths,bedrooms,date_added,dist_centro_km,Total_Area
0,1625,House,258894.27,DHA Defence,Lahore,31.485699,74.473604,7,6,2018-08-05,17.37,708.20
1,4312,House,144758.09,Wapda Town,Lahore,31.436646,74.267135,5,5,2018-08-05,17.21,252.93
2,4191,House,172596.18,State Life Housing Society,Lahore,31.447081,74.396753,7,7,2018-08-05,16.31,505.86


In [29]:
print(df.shape)
print(df.dtypes)
df["city"].value_counts()

(64722, 12)
location_id                int64
property_type             object
price                    float64
location                  object
city                      object
latitude                 float64
longitude                float64
baths                      int64
bedrooms                   int64
date_added        datetime64[ns]
dist_centro_km           float64
Total_Area               float64
dtype: object


city
Karachi       23479
Lahore        22802
Islamabad     10046
Rawalpindi     6669
Faisalabad     1726
Name: count, dtype: int64

In [30]:
Path("../data/silver").mkdir(parents=True, exist_ok=True)
df.to_csv("../data/silver/salida.csv", index=False)
print("Guardado data/silver/salida.csv:", df.shape)

Guardado data/silver/salida.csv: (64722, 12)
